# 3.1 — Precipitation accuracy assessment at the seasonal scale

The analysis repeats the accuracy assessment independently for Fall (October–December), Winter (January–March), Spring (April–June), and Summer (July–September). Each seasonal raster represents the accumulated precipitation for its three-month period.

The analysis calculates MBE, MAE, RMSE, and PBIAS from paired modeled and reference seasonal values. The analysis normalizes the MAE for each season by the multiannual modeled mean for that same season:

$$
\mathrm{nMAE}_s(\%)=100\frac{\mathrm{MAE}_s}{\overline{P}_{\mathrm{model},s}},
$$

where $s$ represents one of the four seasons. This seasonal denominator prevents the annual climatology from masking differences in the seasonal water-balance magnitude.


## Configuration, seasonal labels, and validation

The workflow preserves the source filename suffixes `1230`, `0330`, `0630`, and `0930` for Fall, Winter, Spring, and Summer. The workflow uses calendar-year labels and validated every configured seasonal raster before calculation.


In [ ]:
from pathlib import Path
import sys

import geopandas as gpd

REPOSITORY_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
sys.path.insert(0, str(REPOSITORY_ROOT / "src"))

from accuracy_assessment import (
    annual_pbias,
    compute_metrics,
    enabled_items,
    load_comparison,
    load_config,
    paired_valid,
    validate_inputs,
    write_metric_set,
    write_raster,
    years_for,
)
from regional_plots import plot_grouped_violins

COMPONENT = "precipitation"
SCALE = "seasonal"
config = load_config(REPOSITORY_ROOT / "config.yml")
component_config = config["components"][COMPONENT]
models = enabled_items(component_config, "models")
references = enabled_items(component_config, "references")
regions = gpd.read_file(config["regions"]["file"])
years = years_for(config, SCALE, COMPONENT, assessment="accuracy")
output_dir = Path(config["output_dir"]) / "accuracy" / COMPONENT
output_dir.mkdir(parents=True, exist_ok=True)
print(f"{COMPONENT} {SCALE} period: {years[0]}–{years[-1]}")
seasons = config["seasons"]
validate_inputs(config, SCALE, years, seasons, component_config=component_config)


## Seasonal raster alignment and metrics

The workflow aligns each reference raster to the corresponding modeled grid, forms a pairwise finite and positive mask, and calculates one complete metric set for each model–reference–season combination. It writes each set to a separate seasonal directory so no statistic is shared across seasons.


In [ ]:
positive_only = config["processing"].get("require_positive_values", True)
resampling = config["processing"].get("resampling", "bilinear")
seasonal_paths = {}

for model_key, model in models.items():
    for reference_key, reference in references.items():
        comparison_key = f"{model_key}_vs_{reference_key}"
        seasonal_paths[comparison_key] = {}
        for season, details in seasons.items():
            model_raw, reference_raw = load_comparison(
                model,
                reference,
                SCALE,
                years,
                season=season,
                suffix=details["suffix"],
                resampling=resampling,
            )
            model_paired, reference_paired = paired_valid(
                model_raw,
                reference_raw,
                positive_only=positive_only,
            )
            metrics = compute_metrics(
                model_paired,
                reference_paired,
                model_for_denominator=model_raw,
            )
            seasonal_paths[comparison_key][season] = write_metric_set(
                metrics,
                output_dir / SCALE / comparison_key / season.lower(),
            )
            print("Completed", comparison_key, season)


## Four-season nMAE distributions

The notebook displays the four seasonal nMAE distributions for the configured paper comparison in one grouped climate-region figure. The analysis calculates the displayed mean, median, sample standard deviation, and pixel count after applying the configured nMAE interval.


In [ ]:
region_config = config["regions"]
global_figure_config = config["figures"]
figure_config = global_figure_config["accuracy"][COMPONENT]
model_key = figure_config["seasonal_model"]
reference_key = figure_config["seasonal_reference"]
comparison_key = f"{model_key}_vs_{reference_key}"
nmae_series = [
    (season, seasonal_paths[comparison_key][season]["nmae_percent"])
    for season in seasons
]
nmae_stats = plot_grouped_violins(
    nmae_series,
    regions,
    region_config["name_field"],
    region_config["code_field"],
    output_dir / "figures" / f"seasonal_nmae_violin_{comparison_key}.png",
    output_dir / "tables" / f"seasonal_nmae_regional_statistics_{comparison_key}.csv",
    "nMAE (%)",
    tuple(figure_config["seasonal_nmae_range"]),
    global_figure_config["colors"][:len(nmae_series)],
    dpi=global_figure_config["dpi"],
    exclude_zero=False,
)
nmae_stats


## Four-season PBIAS distributions

The workflow uses the same model–reference pair and climate-region grouping for seasonal PBIAS. Values above zero represent overestimation, values below zero represent underestimation, and exact zero represents no accumulated bias.


In [ ]:
pbias_series = [
    (season, seasonal_paths[comparison_key][season]["pbias"])
    for season in seasons
]
pbias_stats = plot_grouped_violins(
    pbias_series,
    regions,
    region_config["name_field"],
    region_config["code_field"],
    output_dir / "figures" / f"seasonal_pbias_violin_{comparison_key}.png",
    output_dir / "tables" / f"seasonal_pbias_regional_statistics_{comparison_key}.csv",
    "PBIAS (%)",
    tuple(figure_config["seasonal_pbias_range"]),
    global_figure_config["colors"][:len(pbias_series)],
    dpi=global_figure_config["dpi"],
    exclude_zero=False,
)
pbias_stats
